# Mohan Sir — Multilingual Lip-Sync Pilot (Preflight Only)

**Status: PREFLIGHT ONLY — no model inference is implemented in this notebook.** This is a new, isolated pilot intake for an audio-driven talking-video experiment. The target speech scope is English (including the spoken names of A–Z), Hindi, and Marathi. It is not a 3D avatar or live-class engine, and it does not reuse legacy generated clips.

This notebook checks the current runtime and lets you select an authorized source video and its matched speech audio for local inspection. It does **not** install packages, clone repositories, download models/checkpoints, run inference, contact Bunny, or upload files. No output/render is claimed.

The proposed isolated pilot uses MuseTalk with YuNet and an experimental DWPose-landmark-derived mask. Those replacements have not been implemented or validated. S3FD and the BiSeNet face-parsing checkpoint are excluded. Remaining model, dependency, and training-data rights review is still required; see the [MuseTalk GPU Acceptance Report](../../docs/MUSETALK_GPU_ACCEPTANCE_REPORT.md).

## 1. Check the authorized Colab GPU runtime

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another GPU you are authorized to use), then run the next cell. This only inspects the runtime; it does not install anything.

In [ ]:
import importlib.util
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
if importlib.util.find_spec("torch") is None:
    print("PyTorch: not installed; this preflight will not install it.")
else:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA runtime:", torch.version.cuda)

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15, check=False)
    print(result.stdout if result.returncode == 0 else "nvidia-smi returned an error:", result.stderr[:1000])
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("No NVIDIA GPU was confirmed:", str(exc))

## 2. Confirm input authorization

Only proceed if you have permission to process the person and audio in this isolated pilot. The video and audio should be an authorized source recording plus the exact speech audio to synchronize—not an old generated avatar clip. Do not upload private materials to a shared runtime. This notebook keeps selected files in the current Colab session and does not send them to Bunny or another service.

In [ ]:
CONSENT_AND_PROCESSING_AUTHORIZED = False  # Change to True only after confirming permission for this processing.
SOURCE_PROVENANCE_RECORDED = False          # Change to True only after recording the source and audio provenance.

if not CONSENT_AND_PROCESSING_AUTHORIZED or not SOURCE_PROVENANCE_RECORDED:
    raise RuntimeError("Stopped: confirm authorization and provenance above before selecting any media.")

print("Authorization gate confirmed for this session. Keep the evidence with the project owner; do not enter personal details here.")

## 3. Select one source video and matched speech audio

Upload one short MP4 and its matched WAV, MP3, or M4A audio. For the later QA set, include real speech samples in English (spoken A–Z), Hindi, and Marathi. Keep the original audio and the exact spoken text/transcript available for human comparison. This cell only copies the two selected files into this temporary Colab session.

In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab to use the local upload control.") from exc

uploaded = files.upload()
video_candidates = [name for name in uploaded if Path(name).suffix.lower() == ".mp4"]
audio_candidates = [name for name in uploaded if Path(name).suffix.lower() in {".wav", ".mp3", ".m4a"}]
if len(video_candidates) != 1 or len(audio_candidates) != 1 or len(uploaded) != 2:
    raise ValueError("Upload exactly one .mp4 and exactly one .wav, .mp3, or .m4a file; no other files are accepted.")

work_dir = Path("/content/mohan_multilingual_lipsync_pilot")
input_dir = work_dir / "input"
input_dir.mkdir(parents=True, exist_ok=True)
video_path = input_dir / "source.mp4"
audio_path = input_dir / ("speech" + Path(audio_candidates[0]).suffix.lower())
shutil.copyfile(video_candidates[0], video_path)
shutil.copyfile(audio_candidates[0], audio_path)

for path in (video_path, audio_path):
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError("A selected input is missing or empty.")
    print(path.name, "—", path.stat().st_size, "bytes")
print("Inputs are in the temporary Colab session only:", work_dir)

## 4. Read media metadata (no installation)

If `ffprobe` is already available, this reads duration, dimensions, frame rate, and stream types. If it is unavailable, the cell reports that fact and stops; it does not install FFmpeg or access the network.

In [ ]:
import json
import shutil
import subprocess

ffprobe = shutil.which("ffprobe")
if not ffprobe:
    print("ffprobe is unavailable. No packages were installed; inspect media metadata in an approved local tool instead.")
else:
    def probe(path):
        completed = subprocess.run(
            [ffprobe, "-v", "error", "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,avg_frame_rate,sample_rate,channels", "-of", "json", str(path)],
            capture_output=True, text=True, timeout=30, check=True,
        )
        return json.loads(completed.stdout)

    video_info = probe(video_path)
    audio_info = probe(audio_path)
    print("Source video metadata:")
    print(json.dumps(video_info, indent=2))
    print("Speech audio metadata:")
    print(json.dumps(audio_info, indent=2))

## 5. Model and artifact rights gate — intentionally blocked

Before any future inference notebook is built or run, the project owner must verify and record immutable revisions, artifact hashes, licenses/terms, dependency licenses, and relevant training-data provenance for every retained component. MuseTalk, Whisper, the VAE, DWPose, and the proposed YuNet artifact all require artifact-level review. The YuNet detector adapter and geometric mask are not plug-and-play and have not been tested.

**No code below this point downloads, installs, clones, or runs a model. Do not bypass this gate by changing a flag.** When the rights review is complete, create a separately reviewed inference implementation and update this notebook's status.

In [ ]:
MODEL_ARTIFACT_RIGHTS_REVIEW_COMPLETE = False
DEPENDENCY_AND_DATA_PROVENANCE_REVIEW_COMPLETE = False
YUNET_ADAPTER_AND_GEOMETRIC_MASK_VALIDATED = False

if not all((MODEL_ARTIFACT_RIGHTS_REVIEW_COMPLETE, DEPENDENCY_AND_DATA_PROVENANCE_REVIEW_COMPLETE, YUNET_ADAPTER_AND_GEOMETRIC_MASK_VALIDATED)):
    raise RuntimeError("Inference gate is closed. This notebook is preflight-only; do not download or run models.")
raise RuntimeError("This notebook intentionally has no inference implementation. Use a separately reviewed pilot runner after approval.")

## Acceptance checklist for a later, separately approved render

- [ ] Authorized source video and matched speech audio, with provenance and processing consent documented.
- [ ] Actual GPU inference logs, exact command/configuration, immutable model revisions, and SHA-256 hashes recorded.
- [ ] Human QA covers English spoken letter names A–Z, Hindi speech, and Marathi speech; compare against the exact audio, not written spelling alone.
- [ ] Check lip/audio timing, mouth-shape coverage, natural transitions, identity consistency, face boundaries/seams, and frame flicker. Record failures; do not call a passing model run “accurate” without review.
- [ ] Review the rendered file and its audio in real playback at normal speed; retain the source/output pair for comparison.
- [ ] Only after explicit approval, upload the pilot output to Bunny, read it back, and verify playback. Do not use production queues or integrate into the live 3D avatar yet.

**Current result:** none. This notebook only checks runtime/input metadata and does not demonstrate lip-sync quality.